<a href="https://colab.research.google.com/github/joyreuben/AMR-ML-FOR-E.COLI/blob/main/3MTT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd

df = pd.read_csv(
    "/content/spam.csv",
    encoding="latin-1",
    usecols=["v1", "v2"],
)

df.columns = ["label", "message"]
print(f"Rows (messages): {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows (messages): 5572
Columns: 2


In [7]:
counts = df["label"].value_counts()
percentages = df["label"].value_counts(normalize=True) * 100
for label in counts.index:
    print(f"{label:5s}: {counts[label]:5d} messages ({percentages[label]:.1f}%)")

ham  :  4825 messages (86.6%)
spam :   747 messages (13.4%)


In [9]:
df["length"] = df["message"].str.len()
print(df.groupby("label")["length"].describe()[["mean", "min", "max"]])

             mean   min    max
label                         
ham     71.023627   2.0  910.0
spam   138.866131  13.0  224.0


In [10]:
for msg in df[df["label"] == "ham"]["message"].sample(3, random_state=42):
    print(f"- {msg}")

- I am late,so call you tomorrow morning.take care sweet dreams....u and me...ummifying...bye.
- U r too much close to my heart. If u go away i will be shattered. Plz stay with me.
- Wait  &lt;#&gt;  min..


In [11]:
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nDuplicate messages: {df.duplicated(subset='message').sum()}")

Missing values:
label      0
message    0
length     0
dtype: int64

Duplicate messages: 403


In [13]:
import re
import pandas as pd

df = pd.read_csv("/content/spam.csv", encoding="latin-1", usecols=["v1", "v2"])
df.columns = ["label", "message"]
print(f"Loaded {len(df)} messages")

before = len(df)
df = df.drop_duplicates(subset="message").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate messages, {len(df)} remain")

Loaded 5572 messages
Removed 403 duplicate messages, 5169 remain


In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
import joblib

In [19]:
import re
import pandas as pd

# ----------------------------------------------------------------
# 1. Load the raw dataset
# ----------------------------------------------------------------
df = pd.read_csv("/content/spam.csv", encoding="latin-1", usecols=["v1", "v2"])
df.columns = ["label", "message"]
print(f"Loaded {len(df)} messages")

# ----------------------------------------------------------------
# 2. Remove duplicate messages
# ----------------------------------------------------------------
before = len(df)
df = df.drop_duplicates(subset="message").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate messages, {len(df)} remain")


def clean_text(text: str) -> str:
    """Clean a single SMS message while preserving fraud signals."""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " urltoken ", text)
    text = re.sub(r"[£$€₦]|(\bn\d{2,}\b)", " moneytoken ", text)
    text = re.sub(r"\b\d{7,}\b", " phonetoken ", text)
    text = re.sub(r"\b\d+\b", " numtoken ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ----------------------------------------------------------------
# 3. Apply cleaning to every message
# ----------------------------------------------------------------
df["clean_message"] = df["message"].apply(clean_text)

print("\nBEFORE / AFTER EXAMPLES")
for i in df.sample(4, random_state=1).index:
    print(f"\nRAW:   {df.loc[i, 'message']}")
    print(f"CLEAN: {df.loc[i, 'clean_message']}")

# ----------------------------------------------------------------
# 4. Check for empty messages after cleaning
# ----------------------------------------------------------------
empty_after_cleaning = (df["clean_message"].str.strip() == "").sum()
print(f"\nMessages empty after cleaning: {empty_after_cleaning}")

# ----------------------------------------------------------------
# 5. Save the cleaned dataset
# ----------------------------------------------------------------
df.to_csv("/content/spam_clean.csv", index=False)
print(f"\nSaved cleaned dataset to /content/spam_clean.csv ({len(df)} messages)")

Loaded 5572 messages
Removed 403 duplicate messages, 5169 remain

BEFORE / AFTER EXAMPLES

RAW:   I've been barred from all B and Q stores for life!?This twat in orange dungerees came up to me and asked if I wanted decking? So I got the first punch in!!
CLEAN: i ve been barred from all b and q stores for life this twat in orange dungerees came up to me and asked if i wanted decking so i got the first punch in

RAW:   Gam gone after outstanding innings.
CLEAN: gam gone after outstanding innings

RAW:   Am okay. Will soon be over. All the best
CLEAN: am okay will soon be over all the best

RAW:   Welcome! Please reply with your AGE and GENDER to begin. e.g 24M
CLEAN: welcome please reply with your age and gender to begin e g m

Messages empty after cleaning: 2

Saved cleaned dataset to /content/spam_clean.csv (5169 messages)


In [20]:
df = pd.read_csv("/content/spam_clean.csv")

df = df.dropna(subset=["clean_message"])
df = df[df["clean_message"].str.strip() != ""]
print(f"{len(df)} messages after dropping empties")

5167 messages after dropping empties


In [21]:
df["label_num"] = (df["label"] == "spam").astype(int)

from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_message"],
    df["label_num"],
    test_size=0.2,
    stratify=df["label_num"],
    random_state=42,
)

print(f"Train set: {len(X_train_text)} messages")
print(f"Test set:  {len(X_test_text)} messages")

Train set: 4133 messages
Test set:  1034 messages


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=3000,
)

X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

print(f"TF-IDF vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Train matrix shape: {X_train_tfidf.shape}")
print(f"Test matrix shape:  {X_test_tfidf.shape}")

TF-IDF vocabulary size: 3000
Train matrix shape: (4133, 3000)
Test matrix shape:  (1034, 3000)


In [23]:
train_len = df.loc[X_train_text.index, "message"].str.len().values.reshape(-1, 1)
test_len = df.loc[X_test_text.index, "message"].str.len().values.reshape(-1, 1)

from scipy.sparse import hstack, csr_matrix

X_train_final = hstack([X_train_tfidf, csr_matrix(train_len)])
X_test_final = hstack([X_test_tfidf, csr_matrix(test_len)])

print(f"Final train matrix shape (TF-IDF + length): {X_train_final.shape}")
print(f"Final test matrix shape (TF-IDF + length):  {X_test_final.shape}")

Final train matrix shape (TF-IDF + length): (4133, 3001)
Final test matrix shape (TF-IDF + length):  (1034, 3001)


In [24]:
import joblib

joblib.dump(vectorizer, "/content/tfidf_vectorizer.joblib")
joblib.dump(X_train_final, "/content/X_train.joblib")
joblib.dump(X_test_final, "/content/X_test.joblib")
joblib.dump(y_train, "/content/y_train.joblib")
joblib.dump(y_test, "/content/y_test.joblib")

print("Saved vectorizer and train/test feature matrices")

Saved vectorizer and train/test feature matrices


In [25]:
import joblib

X_train = joblib.load("/content/X_train.joblib")
X_test = joblib.load("/content/X_test.joblib")
y_train = joblib.load("/content/y_train.joblib")
y_test = joblib.load("/content/y_test.joblib")

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (4133, 3001), Test: (1034, 3001)


In [27]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Linear SVM": LinearSVC(class_weight="balanced"),
}

In [28]:
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"{name}: trained")

Naive Bayes: trained
Logistic Regression: trained
Linear SVM: trained


In [29]:
import joblib

joblib.dump(trained_models, "/content/trained_models.joblib")
print("Saved trained models")

Saved trained models


In [30]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

for name, model in trained_models.items():
    y_pred = model.predict(X_test)

    print("=" * 60)
    print(name)
    print("=" * 60)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
    print(classification_report(y_test, y_pred, target_names=["ham", "spam"]))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print()

Naive Bayes
Accuracy: 0.974
              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       903
        spam       1.00      0.79      0.89       131

    accuracy                           0.97      1034
   macro avg       0.99      0.90      0.94      1034
weighted avg       0.97      0.97      0.97      1034

Confusion matrix:
[[903   0]
 [ 27 104]]

Logistic Regression
Accuracy: 0.972
              precision    recall  f1-score   support

         ham       0.99      0.98      0.98       903
        spam       0.85      0.94      0.89       131

    accuracy                           0.97      1034
   macro avg       0.92      0.96      0.94      1034
weighted avg       0.97      0.97      0.97      1034

Confusion matrix:
[[882  21]
 [  8 123]]

Linear SVM
Accuracy: 0.986
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       903
        spam       0.95      0.95      0.95       131

    acc

In [31]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.metrics import make_scorer, f1_score

param_grid = {
    "C": [0.01, 0.1, 0.5, 1, 5, 10]
}

scorer = make_scorer(f1_score, pos_label=1)

svm_search = GridSearchCV(
    LinearSVC(class_weight="balanced"),
    param_grid,
    scoring=scorer,
    cv=5,
)

svm_search.fit(X_train, y_train)

print("Best C:", svm_search.best_params_)
print("Best cross-validation F1 (spam class):", svm_search.best_score_)

Best C: {'C': 1}
Best cross-validation F1 (spam class): 0.9464550564738913


In [32]:
best_svm = svm_search.best_estimator_

from sklearn.metrics import classification_report, confusion_matrix

y_pred_best = best_svm.predict(X_test)
print(classification_report(y_test, y_pred_best, target_names=["ham", "spam"]))
print(confusion_matrix(y_test, y_pred_best))

              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       903
        spam       0.95      0.95      0.95       131

    accuracy                           0.99      1034
   macro avg       0.97      0.97      0.97      1034
weighted avg       0.99      0.99      0.99      1034

[[896   7]
 [  7 124]]


In [33]:
test_messages = df.loc[X_test_text.index, "message"].values

false_negatives = [
    msg for msg, true, pred in zip(test_messages, y_test, y_pred_best)
    if true == 1 and pred == 0
]

print(f"Missed fraud messages: {len(false_negatives)}")
for msg in false_negatives:
    print(f"- {msg}")

Missed fraud messages: 7
- Babe: U want me dont u baby! Im nasty and have a thing 4 filthyguys. Fancy a rude time with a sexy bitch. How about we go slo n hard! Txt XXX SLO(4msgs)
- SMS. ac sun0819 posts HELLO:\You seem cool
- Hi its LUCY Hubby at meetins all day Fri & I will B alone at hotel U fancy cumin over? Pls leave msg 2day 09099726395 Lucy x Callså£1/minMobsmoreLKPOBOX177HP51FL
- Would you like to see my XXX pics they are so hot they were nearly banned in the uk!
- In The Simpsons Movie released in July 2007 name the band that died at the start of the film? A-Green Day, B-Blue Day, C-Red Day. (Send A, B or C)
- Check Out Choose Your Babe Videos @ sms.shsex.netUN fgkslpoPW fgkslpo
- Missed call alert. These numbers called but left no message. 07008009200


In [34]:
import joblib
joblib.dump(best_svm, "/content/final_svm_model.joblib")
print("Saved final tuned SVM model")

Saved final tuned SVM model


In [35]:
print("Best C:", svm_search.best_params_)
print("Best cross-validation F1 (spam class):", svm_search.best_score_)

Best C: {'C': 1}
Best cross-validation F1 (spam class): 0.9464550564738913


In [36]:
false_negatives = [
    msg for msg, true, pred in zip(test_messages, y_test, y_pred_best)
    if true == 1 and pred == 0
]
print(f"Missed fraud messages: {len(false_negatives)}")
for msg in false_negatives:
    print(f"- {msg}")

Missed fraud messages: 7
- Babe: U want me dont u baby! Im nasty and have a thing 4 filthyguys. Fancy a rude time with a sexy bitch. How about we go slo n hard! Txt XXX SLO(4msgs)
- SMS. ac sun0819 posts HELLO:\You seem cool
- Hi its LUCY Hubby at meetins all day Fri & I will B alone at hotel U fancy cumin over? Pls leave msg 2day 09099726395 Lucy x Callså£1/minMobsmoreLKPOBOX177HP51FL
- Would you like to see my XXX pics they are so hot they were nearly banned in the uk!
- In The Simpsons Movie released in July 2007 name the band that died at the start of the film? A-Green Day, B-Blue Day, C-Red Day. (Send A, B or C)
- Check Out Choose Your Babe Videos @ sms.shsex.netUN fgkslpoPW fgkslpo
- Missed call alert. These numbers called but left no message. 07008009200


In [37]:
import joblib
joblib.dump(best_svm, "/content/final_svm_model.joblib")
print("Saved final tuned SVM model")

Saved final tuned SVM model
